# Zadání první semestrální práce

Ve trojrozměrném prostoru máme osm nepohybujících se mikrofonů s polohami uloženými v proměnné ```mics```.

Mezi těmito mikrofony se pohybuje objekt vydávající zvuk. Zvuky zachycené jednotlivými mikrofony jsou uložené v poli ```sounds```. První rozměr tohoto pole je index mikrofonu a druhý rozměr je vzorek zvuku.

Objekt vydává periodický zvuk o frekvenci ```f_sound```.

Rychlost zvuku je konstantní a je známá a uložena v proměnné ```v_sound```. Jednotky jsou metry za sekundu.

Následuje generování zvuků.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

np.random.seed(123)

mics = np.asarray([
    [[-1, -1, -1]],
    [[-1, -1, +1]],
    [[-1, +1, -1]],
    [[-1, +1, +1]],
    [[+1, -1, -1]],
    [[+1, -1, +1]],
    [[+1, +1, -1]],
    [[+1, +1, +1]],
], dtype=np.float64)  # shape: (n_mics, 1, dim)

n_mics = mics.shape[0]
T = 100  # [s]
f_sample = 44_100  # [Hz]
v_sound = 340  # [m/s]
f_sound = 110  # [Hz]

t = np.arange(0, T, 1 / f_sample)  # shape: (N, )
trajectory = 0.5 * np.asarray([
    np.sin(2 * np.pi * t * 2 / T),
    np.cos(2 * np.pi * t * 2 / T),
    t / T,
], dtype=np.float64).T  # shape: (N, dim)

def generate_sound(t):
    return np.sign(np.sin(2 * np.pi * f_sound * t) - 0.99)

relative_positions = trajectory[None, ...] - mics  # shape: (n_mics, n_samples, dim)
distances = np.linalg.norm(relative_positions, axis=-1)  # shape: (n_mics, n_samples)
delays = distances / v_sound  # shape: (n_mics, n_samples)
gains = 1 / (1 + 2 * distances**2)  # shape: (n_mics, n_samples)
sounds = np.asarray([
    gains[i] * generate_sound(t - delays[i]) + 0.01 * np.random.randn(len(t))
    for i in range(n_mics)
], dtype=np.float64)  # shape: (n_mics, n_samples)

# Plot first 1000 samples for each channel
fig, axs = plt.subplots(n_mics, 1, figsize=(10, 6), sharex=True)
for i_mic in range(n_mics):
    axs[i_mic].plot(t[:1000], sounds[i_mic, :1_000])
    axs[i_mic].set_ylabel(f'Mik. {i_mic + 1}')
    axs[i_mic].set_ylim(-1.0, 1.0)
axs[-1].set_xlabel("čas [s]")
fig.suptitle("Signály zaznamenané mikrofonovým polem")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib import pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2])
for i_mic in range(n_mics):
    ax.plot(mics[i_mic, 0, 0], mics[i_mic, 0, 1], mics[i_mic, 0, 2], 'o', color="red")
plt.show()


## 1. úkol

Rozdělte pro každý mikrofon signál na kratší segmenty. Délku i posun si zvolte sami tak, aby to dovolovalo splnit následující úkoly.

In [ ]:
def do_segmentation(x, window_length, stride):
    ...


window = ...
stride = ...

sounds_segmented = do_segmentation(sounds, window, stride)  # shape: (n_mics, n_segments, window)

## 2. úkol

Pro každou dvojici mikrofonů a pro každý segment odhadněte časový posun. Pro dvojice, ve kterých se jedná o ten samý mikrofon, odhadovat zpoždění nemusíte. Můžete bez odhadu použít hodnotu nula sekund.

Pozor: frekvence zvuku ```f_sound``` a vzorkovací frekvence ```f_sample``` určuje, o kolik vzorků maximálně jsme schopni odhadnout zpoždění. Použijte tuto úvahu a zakažte algoritmu odhadovat větší zpoždění než maximálně možné.


In [ ]:
delays_differences_estimation = ...  # shape: (n_mics, n_mics, n_segments)

Ověřte, že správně odhadujete rozdíly i pro záporné hodnoty.

In [ ]:
real_delays_difference_0_2 = delays[0, ::stride] - delays[2, ::stride]

plt.plot(delays_differences_estimation[0, 2, :], label="odhad")
plt.plot(real_delays_difference_0_2, label="skutečnost")
plt.xlabel("Index segmentu")
plt.ylabel(r'$\Delta delays$ [s]')
plt.title('Odhad rozdílů zpoždění a skutečné rozdíly zpoždění')
plt.legend()
plt.show()


## 3. úkol

Použijte rychlost zvuku a přepočtěte rozdíly časů na rozdíly poloh. Z rozdílu poloh a pomocí známých poloh mikrofonů v mikrofonovém poli odhadněte absolutní polohu objektu v čase. Stačí jedna poloha pro každý segment. Využijte nějakou metodu hledající přibližné řešení.

In [ ]:
relative_positions_differences_estimation = ...

...

Vykreslete nalezené řešení.

In [ ]:
trajectory_estimated = ...
mics_estimated = mics

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(trajectory_estimated[:, 0], trajectory_estimated[:, 1], trajectory_estimated[:, 2])
for i_mic in range(n_mics):
    ax.plot(mics_estimated[i_mic, 0, 0], mics_estimated[i_mic, 0, 1], mics_estimated[i_mic, 0, 2], 'o', color="red")
plt.title('Odhadnutá trajektorie')
plt.show()

Porovnejte se skutečnou trajektorií.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2])
for i_mic in range(n_mics):
    ax.plot(mics[i_mic, 0, 0], mics[i_mic, 0, 1], mics[i_mic, 0, 2], 'o', color="red")
plt.title('Skutečná trajektorie')
plt.show()

## 4. úkol

Považujte polohy mikrofonů za neměnné, avšak neznámé. Odhadněte najednou nejen trajektorii, ale i polohy mikrofonů. Vykreslete nalezené řešení.

Výsledné řešení nemusí odpovídat původním souřadnicím. Za správné považujte i řešení, které se od skutečného řešení liší posunutím, otočením nebo zrcadlením souřadnicového systému.

In [ ]:
...

## Dobrovolné úkoly nad rámec zadání

...

## Závěr

...